congestion model

In [0]:
%sql
use catalog sagar_cap3_cat1;
CREATE SCHEMA IF NOT EXISTS models;
use catalog sagar_cap3_cat1;
USE SCHEMA models;


In [0]:
from pyspark.sql.functions import col
top_intersections = spark.table("sagar_cap3_cat1.silver.intersections").filter(col("is_top50") == True)
df=spark.table('sagar_cap3_cat1.silver.events_data')
top_intersections_stream = df.join(top_intersections, on="intersection_id", how="inner")
display(top_intersections_stream)


intersection_id,sensor_id,zone,corridor_id,lane_status,speed_unit,vehicle_count,aqi,occupancy,battery_level,avg_speed_kmh,event_ts,ingest_ts,geo,name,corridor_id,corridor_name,zone,latitude,longitude,school_distance_m,hospital_distance_m,near_school,near_hospital,is_top50
X00139,S00024,NORTHWEST,C002,OPEN,km/h,4,38,0.62,0.86,62.45,2025-08-29T01:00:30.000Z,2025-08-29T01:00:33.000Z,"List(28.38, 77.35)",Junction 139,C002,Inner Ring,NORTHWEST,28.38,77.35,null,null,false,false,true
X00812,S00558,NORTHWEST,C002,OPEN,km/h,4,40,0.62,0.86,59.74,2025-08-29T01:17:00.000Z,2025-08-29T01:17:15.000Z,"List(28.75, 77.43)",Junction 812,C002,Inner Ring,NORTHWEST,28.75,77.43,null,null,false,false,true
X00346,S00119,NORTH,C004,OPEN,km/h,4,48,0.62,0.86,42.47,2025-08-29T02:18:00.000Z,2025-08-29T02:18:07.000Z,"List(28.82, 77.44)",Junction 346,C004,Gurgaon Expwy,NORTH,28.82,77.44,null,null,false,false,true
X00577,S00318,WEST,C006,OPEN,km/h,2,45,0.62,0.86,47.68,2025-08-29T02:30:30.000Z,2025-08-29T02:30:55.000Z,"List(28.71, 77.41)",Junction 577,C006,Tech Park Link,WEST,28.71,77.41,null,null,false,false,true
X00036,S00107,NORTH,C004,OPEN,km/h,2,43,0.62,0.86,56.13,2025-08-29T02:35:00.000Z,2025-08-29T02:35:12.000Z,"List(28.62, 77.02)",Junction 36,C004,Gurgaon Expwy,NORTH,28.62,77.02,null,null,false,false,true
X00332,S00036,WEST,C006,OPEN,km/h,3,48,0.62,0.86,59.91,2025-08-29T02:36:30.000Z,2025-08-29T02:36:42.000Z,"List(28.75, 77.21)",Junction 332,C006,Tech Park Link,WEST,28.75,77.21,193,null,true,false,true
X00812,S00114,NORTHWEST,C002,OPEN,km/h,4,53,0.62,0.86,35.21,2025-08-29T03:04:30.000Z,2025-08-29T03:04:50.000Z,"List(28.75, 77.43)",Junction 812,C002,Inner Ring,NORTHWEST,28.75,77.43,null,null,false,false,true
X00185,S00124,CENTRAL,C009,OPEN,km/h,4,38,0.62,0.86,64.73,2025-08-29T03:39:00.000Z,2025-08-29T03:39:21.000Z,"List(28.6, 77.09)",Junction 185,C009,Harbor Express,CENTRAL,28.6,77.09,543,null,true,false,true
X00649,S00313,CENTRAL,C005,OPEN,km/h,4,45,0.62,0.86,54.31,2025-08-29T04:07:00.000Z,2025-08-29T04:07:15.000Z,"List(28.56, 77.39)",Junction 649,C005,Noida Link,CENTRAL,28.56,77.39,400,null,true,false,true
X00346,S00119,NORTH,C004,OPEN,km/h,2,37,0.62,0.86,61.29,2025-08-29T04:18:30.000Z,2025-08-29T04:18:53.000Z,"List(28.82, 77.44)",Junction 346,C004,Gurgaon Expwy,NORTH,28.82,77.44,null,null,false,false,true


In [0]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

# Prepare data
pdf = top_intersections_stream.toPandas()
X = pdf[["vehicle_count"]]
y = pdf["avg_speed_kmh"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
model = RandomForestRegressor(n_estimators=100, max_depth=5)
model.fit(X_train, y_train)

# Infer model signature (input/output schema)
signature = infer_signature(X_train, model.predict(X_train))

with mlflow.start_run():
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="model",
        registered_model_name="sagar_cap3_cat1.models.congestion_forecast",
        signature=signature,
        input_example=X_train.iloc[:5]   # optional but recommended
    )


/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
Registered model 'sagar_cap3_cat1.models.congestion_forecast' already exists. Creating a new version of this model...
Created version '1' of model 'sagar_cap3_cat1.models.congestion_forecast'.


In [0]:
from mlflow import MlflowClient
client = MlflowClient()

for mv in client.search_model_versions("name='sagar_cap3_cat1.models.congestion_forecast'"):
    print(f"Version: {mv.version}, Stage: {mv.current_stage}")


Version: 1, Stage: None


In [0]:
import mlflow
import mlflow.pyfunc
import pandas as pd

# Sample data (replace this with your own test data)
data = {
    "vehicle_count": [100, 200, 300]
}
df = pd.DataFrame(data).astype({'vehicle_count': 'int32'})

# Model name 
model_name = "congestion_forecast"

# Load latest version of the model
model = mlflow.pyfunc.load_model(f"models:/{model_name}/1")

# Run prediction
predictions = model.predict(df)

print("Predictions:")
print(predictions)


Predictions:
[55.71514768 55.71514768 55.71514768]


 support champion/challenger comparisons and drift monitoring